# Task 3: Transfer Learning and Fine-Tuning



In [8]:
# to run on colab, simply execute the following commands in a cell:
# !git clone https://github.com/Torpedoooo/deeplearning-assignment-1.git
# !pip install torch torchvision pandas seaborn scikit-learn
# !pip install nbconvert
# %cd deeplearning-assignment-1/
# !jupyter nbconvert --execute task3.ipynb \
#                     --to notebook --output executed.ipynb

In [9]:
import os
import time
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.models import ResNet18_Weights


In [10]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cpu


In [11]:
class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        classes = sorted(self.img_labels.iloc[:, 1].unique())
        self.class2idx = {c: i for i, c in enumerate(classes)}
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_name = self.img_labels.iloc[idx, 0]
        if not img_name.lower().endswith('.png'):
            img_name = img_name + '.png'
        label = self.img_labels.iloc[idx, 1]
        label = self.class2idx[label]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)

        return image, label


In [12]:
df = pd.read_csv('train_labels.csv')
train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)

train_df.to_csv('train_split.csv', index=False)
val_df.to_csv('val_split.csv', index=False)

print('train:', len(train_df), 'rows')
print('val  :', len(val_df), 'rows')

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15), ratio=(0.3, 3.3), value='random')
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


train: 2880 rows
val  : 720 rows


In [13]:
train_dir = './Train/'
train_ds = CustomImageDataset('train_split.csv', train_dir, transform=train_transform)
val_ds = CustomImageDataset('val_split.csv', train_dir, transform=val_transform)

labels = train_ds.img_labels['label'].map(train_ds.class2idx)
class_counts = labels.value_counts().sort_index()
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()

sample_weights = torch.tensor(labels.map(class_weights).values, dtype=torch.double)
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=64, sampler=sampler, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2)

print('class counts:')
print(class_counts)


class counts:
label
0    294
1    227
2    306
3    240
4    192
5    488
6    369
7    214
8    550
Name: count, dtype: int64


### Model selection and justification

- `ResNet18` because it is a compact, stable pretrained model with good feature transfer ability.
- It is appropriate for an image classification task with moderate dataset size.
- Its residual structure helps fine-tuning without vanishing gradients.


In [ ]:
model = models.resnet18(weights=ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 9) # 9 classes in our dataset
model = model.to(device)
print(model)


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [15]:
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

optimizer = optim.AdamW(model.fc.parameters(), lr=5e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)
criterion = nn.CrossEntropyLoss()

def batch_accuracy(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    running_acc = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            running_loss += loss.item() * xb.size(0)
            running_acc += batch_accuracy(logits, yb) * xb.size(0)
            all_preds.append(logits.argmax(dim=1).cpu())
            all_targets.append(yb.cpu())
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    loss = running_loss / len(loader.dataset)
    acc = running_acc / len(loader.dataset)
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    return loss, acc, macro_f1

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss = 0.0
    running_acc = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
        running_acc += batch_accuracy(logits, yb) * xb.size(0)
    loss = running_loss / len(loader.dataset)
    acc = running_acc / len(loader.dataset)
    return loss, acc


In [ ]:
best_f1 = 0.0
patience = 12
trigger = 0
epochs = 8

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
    scheduler.step(val_f1)

    if val_f1 > best_f1:
        best_f1 = val_f1
        trigger = 0
        torch.save(model.state_dict(), 'best_task3_stage1.pth')
        print(f'Stage1 Epoch {epoch}: new best val_f1 {val_f1:.4f}')
    else:
        trigger += 1

    print(f'Stage1 Epoch {epoch} | train loss {train_loss:.4f} acc {train_acc:.4f} '
          f'| val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} '
          f'(patience {trigger}/{patience})')
    if trigger >= patience:
        break


In [ ]:
for name, param in model.named_parameters():
    if 'layer4' in name or 'fc' in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=4)

best_f1_stage2 = 0.0
trigger = 0
epochs = 20

for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
    scheduler.step(val_f1)

    if val_f1 > best_f1_stage2:
        best_f1_stage2 = val_f1
        trigger = 0
        torch.save(model.state_dict(), 'best_task3_finetuned.pth')
        print(f'Stage2 Epoch {epoch}: new best val_f1 {val_f1:.4f}')
    else:
        trigger += 1

    print(f'Stage2 Epoch {epoch} | train loss {train_loss:.4f} acc {train_acc:.4f} '
          f'| val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} '
          f'(patience {trigger}/{patience})')
    if trigger >= patience:
        break


In [ ]:
model.load_state_dict(torch.load('best_task3_finetuned.pth'))
model.eval()

val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
print(f'Final best model: val loss {val_loss:.4f}, val acc {val_acc:.4f}, val macro F1 {val_f1:.4f}')

all_preds, all_targets = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(yb.cpu())

all_preds = torch.cat(all_preds)
all_targets = torch.cat(all_targets)

type_names = sorted(train_ds.class2idx, key=lambda k: train_ds.class2idx[k])
print(classification_report(all_targets, all_preds, target_names=type_names))

cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=type_names, yticklabels=type_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('ResNet18 Validation Confusion Matrix')
plt.show()


In [ ]:
idx2class = {v: k for k, v in train_ds.class2idx.items()}
test_ids = sorted([f.split('.png')[0] for f in os.listdir('Test') if f.endswith('.png')])

class ImageOnlyDataset(Dataset):
    def __init__(self, ids, img_dir, transform=None):
        self.ids = ids
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        name = self.ids[idx]
        path = os.path.join(self.img_dir, name + '.png')
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image

submission_ds = ImageOnlyDataset(test_ids, 'Test/', transform=val_transform)
submission_loader = DataLoader(submission_ds, batch_size=64, shuffle=False, num_workers=2)

model.eval()
submission_preds = []
with torch.no_grad():
    for xb in submission_loader:
        xb = xb.to(device)
        logits = model(xb)
        submission_preds.extend(logits.argmax(dim=1).cpu().tolist())

assert len(submission_preds) == len(test_ids)
out_df = pd.DataFrame({'Id': test_ids, 'label': [idx2class[i] for i in submission_preds]})
out_df.to_csv('submission_task3.csv', index=False)
print('Saved submission_task3.csv')
